In [ ]:
# ============================================
# INSTALL & IMPORT LIBRARIES
# ============================================
!pip install pandas nltk scikit-learn tensorflow

import pandas as pd
import numpy as np
import re
import nltk

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ============================================
# LOAD DATASET
# ============================================
from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]

df = pd.read_csv(filename, encoding='latin-1')

df.columns = df.columns.str.strip().str.lower()

text_col = None
for col in df.columns:
    if 'response' in col or 'feedback' in col or 'text' in col:
        text_col = col
        break

# ============================================
# PREPROCESSING
# ============================================
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()

    words = [
        lemmatizer.lemmatize(w)
        for w in words
        if w not in stop_words and len(w) > 2
    ]

    return " ".join(words)

df['cleaned'] = df[text_col].apply(clean_text)

# ============================================
# LABELING
# ============================================
keywords = {
    'Appreciation': ['thank','grateful','appreciate','thankful','good','great','satisfied'],
    'Challenges': ['problem','issue','delay','error','difficult','hard','slow'],
    'Suggestion': ['should','suggest','recommend','improve','better']
}

def assign_labels(text):
    labels = []

    for category, words in keywords.items():
        if any(w in text for w in words):
            labels.append(category)

    if not labels:
        labels = ['Neutral']

    return labels

df['labels'] = df['cleaned'].apply(assign_labels)

# ============================================
# FEATURE EXTRACTION
# ============================================
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1,2)
)

X = vectorizer.fit_transform(df['cleaned']).toarray()

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['labels'])

# ============================================
# TRAIN TEST SPLIT
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

# ============================================
# MODEL
# ============================================
model = Sequential([
    Dense(128, activation='relu', input_shape=(X.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(len(mlb.classes_), activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

# ============================================
# TRAINING
# ============================================
model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=16,
    validation_data=(X_test, y_test)
)

# ============================================
# EVALUATION
# ============================================
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs >= 0.3).astype(int)

print("F1 Micro:", f1_score(y_test, y_pred, average='micro'))
print("F1 Macro:", f1_score(y_test, y_pred, average='macro'))

print(classification_report(y_test, y_pred, target_names=mlb.classes_))

# ============================================
# PREDICTION
# ============================================
def predict(text, threshold=0.3):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned]).toarray()

    probs = model.predict(vec)[0]

    labels = [
        mlb.classes_[i]
        for i, p in enumerate(probs)
        if p >= threshold
    ]

    if not labels:
        labels = ["Neutral"]

    return labels, probs

def show_percentages(probs):
    for label, prob in zip(mlb.classes_, probs):
        print(f"{label}: {prob*100:.2f}%")

while True:
    x = input("Enter text (or exit): ")

    if x.lower() == 'exit':
        break

    labels, probs = predict(x)

    print("Predicted Labels:", labels)
    show_percentages(probs)